# Tugas 2 - Text Preprocessing, TF-IDF, ICSDF, dan PCA

## 2. Import Library

Library yang digunakan untuk membaca dan mengolah dataset, melakukan preprocessing teks, pembagian data, pembobotan TF-IDF, perhitungan ICSDF, serta reduksi dimensi menggunakan PCA.

In [1]:
!pip install -q pandas numpy scipy scikit-learn openpyxl

In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

print("Semua library berhasil diimport.")

Semua library berhasil diimport.


## 3. Membaca Dataset

Dataset yang digunakan merupakan kumpulan 200 berita dari Detik.com yang terdiri dari dua kategori, yaitu Sport dan Finance.

Dataset memiliki tiga kolom utama:

- `id` sebagai identitas berita
- `isi_berita` sebagai isi teks berita
- `label` sebagai kategori berita

In [3]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

print("Jumlah data :", len(df))

print("\nNama kolom :")
print(df.columns)

print("\nJumlah data per label :")
print(df["label"].value_counts())

display(df.head())

Jumlah data : 200

Nama kolom :
Index(['id', 'isi_berita', 'label'], dtype='str')

Jumlah data per label :
label
sport      100
finance    100
Name: count, dtype: int64


,id,isi_berita,label
0,1,Menpora Erick Thohir senang kontingen Indonesi...,sport
1,2,Rangkaian seri balap MotoGP Jepang di Sirkuit ...,sport
2,3,Jorge Martin sedang memperjuangkan titel juara...,sport
3,4,Rider Indonesia Veda Ega Pratama balapan di Ma...,sport
4,5,"Pelatih speed tim panjat tebing, Fitriyani, me...",sport


## 4. Mengubah Label Menjadi Numerik

Label teks diubah menjadi bentuk numerik agar dapat digunakan dalam proses pengolahan data.

Aturan yang digunakan:

- `sport` menjadi `1`
- `finance` menjadi `0`

In [4]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

display(
    df[["id", "label", "label_num"]].head()
)

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


## 5. Menghitung Jumlah Kata Semua Berita

Sebelum dilakukan preprocessing, jumlah kata pada setiap berita dihitung terlebih dahulu.

Perhitungan dilakukan dengan memisahkan teks berdasarkan spasi menggunakan fungsi `split()`.

Hasil perhitungan disimpan pada kolom `jumlah_kata_asli`.

In [5]:
df["jumlah_kata_asli"] = (
    df["isi_berita"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print("Total seluruh kata :", df["jumlah_kata_asli"].sum())

display(
    df[["id", "label", "jumlah_kata_asli"]].head()
)

Total seluruh kata : 68169


,id,label,jumlah_kata_asli
0,1,sport,279
1,2,sport,228
2,3,sport,223
3,4,sport,247
4,5,sport,400


## 6. Kamus Kata Tidak Baku

Kamus kata tidak baku digunakan untuk mengubah kata singkatan atau kata tidak baku menjadi bentuk yang lebih standar.

Contoh:

- `yg` menjadi `yang`
- `dgn` menjadi `dengan`
- `utk` menjadi `untuk`
- `krn` menjadi `karena`
- `tdk` menjadi `tidak`
- `dr` menjadi `dari`

In [6]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

print("Kamus kata tidak baku berhasil dibuat.")
print("Jumlah kata :", len(kamus_tidak_baku))

Kamus kata tidak baku berhasil dibuat.
Jumlah kata : 28


## 7. Kamus Bahasa Asing

Kamus bahasa asing digunakan untuk menormalisasi beberapa istilah asing yang terdapat pada berita menjadi istilah bahasa Indonesia.

Normalisasi disesuaikan dengan dua kategori dataset, yaitu Sport dan Finance.

In [7]:
kamus_asing = {
    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

print("Kamus bahasa asing berhasil dibuat.")
print("Jumlah istilah :", len(kamus_asing))

Kamus bahasa asing berhasil dibuat.
Jumlah istilah : 38


## 8. Fungsi Preprocessing

Tahapan preprocessing yang dilakukan adalah:

1. Mengubah teks menjadi string
2. Case folding
3. Menghapus URL
4. Menghapus alamat email
5. Menghapus angka
6. Menghapus tanda baca, simbol, dan emotikon
7. Menghapus spasi berlebih
8. Tokenisasi
9. Normalisasi kata tidak baku
10. Normalisasi istilah bahasa asing
11. Menggabungkan kembali token menjadi teks

In [8]:
def preprocessing(text):
    text = str(text)

    # Case folding
    text = text.lower()

    # Hapus URL
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )

    # Hapus email
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    # Hapus angka
    text = re.sub(
        r'\d+',
        ' ',
        text
    )

    # Hapus tanda baca, simbol, dan emotikon
    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )

    # Hapus spasi berlebih
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # Tokenisasi
    tokens = text.split()

    # Normalisasi kata tidak baku dan bahasa asing
    hasil = []

    for token in tokens:
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]

        if token in kamus_asing:
            token = kamus_asing[token]

        hasil.append(token)

    return " ".join(hasil)

print("Fungsi preprocessing berhasil dibuat.")

Fungsi preprocessing berhasil dibuat.


## 9. Menerapkan Preprocessing ke Semua Berita

Fungsi preprocessing diterapkan pada seluruh isi berita.

Hasil preprocessing disimpan pada kolom baru bernama `berita_clean`.

In [9]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

display(
    df[["isi_berita", "berita_clean"]].head()
)

,isi_berita,berita_clean
0,Menpora Erick Thohir senang kontingen Indonesi...,menpora erick thohir senang kontingen indonesi...
1,Rangkaian seri balap MotoGP Jepang di Sirkuit ...,rangkaian seri balap motogp jepang di sirkuit ...
2,Jorge Martin sedang memperjuangkan titel juara...,jorge martin sedang memperjuangkan titel juara...
3,Rider Indonesia Veda Ega Pratama balapan di Ma...,pembalap indonesia veda ega pratama balapan di...
4,"Pelatih speed tim panjat tebing, Fitriyani, me...",pelatih speed tim panjat tebing fitriyani meng...


## 10. Menghitung Jumlah Kata Setelah Preprocessing

Setelah teks dibersihkan melalui preprocessing, jumlah kata pada setiap berita dihitung kembali.

Hasil perhitungan disimpan pada kolom `jumlah_kata_clean`.

In [10]:
df["jumlah_kata_clean"] = (
    df["berita_clean"]
    .apply(lambda x: len(x.split()))
)

display(
    df[
        [
            "id",
            "label",
            "jumlah_kata_asli",
            "jumlah_kata_clean"
        ]
    ].head()
)

,id,label,jumlah_kata_asli,jumlah_kata_clean
0,1,sport,279,286
1,2,sport,228,229
2,3,sport,223,221
3,4,sport,247,232
4,5,sport,400,391


## 11. Mengekstrak Seluruh Kata Unik

Setelah seluruh berita melalui proses preprocessing, semua kata dari setiap berita dikumpulkan.

Kata-kata tersebut kemudian diubah menjadi kumpulan kata unik menggunakan `set()`.

Tahap ini digunakan untuk mengetahui jumlah seluruh kata dan jumlah kata unik setelah preprocessing.

In [11]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(berita.split())

kata_unik = sorted(set(semua_kata))

print("Jumlah seluruh kata :", len(semua_kata))
print("Jumlah kata unik :", len(kata_unik))

print("\nContoh kata unik:")
print(kata_unik[:50])

Jumlah seluruh kata : 64711
Jumlah kata unik : 7279

Contoh kata unik:
['a', 'aadi', 'abad', 'abal', 'abang', 'abdi', 'abdul', 'abdullah', 'abdurrahman', 'abe', 'abipraya', 'abruzzo', 'absen', 'abu', 'abuan', 'ac', 'academy', 'acara', 'acaranya', 'acas', 'acceleration', 'accountant', 'accounting', 'accuracy', 'acd', 'aceh', 'aci', 'acinya', 'acosta', 'action', 'acuan', 'acuannya', 'acungi', 'ad', 'ada', 'adakan', 'adalah', 'adanya', 'adapun', 'addict', 'ade', 'adeliya', 'adhi', 'adi', 'adianto', 'adief', 'adik', 'adil', 'aditya', 'admin']


## 12. Membagi Data Training dan Testing

Dataset dibagi menjadi:

- 160 data training
- 40 data testing

`random_state=42` digunakan agar pembagian data dapat direproduksi.

Parameter `stratify` digunakan agar distribusi label tetap seimbang pada data training dan testing.

In [12]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["berita_clean"],
    df["label_num"],
    test_size=40,
    random_state=42,
    stratify=df["label_num"]
)

print("Jumlah training :", len(X_train_text))
print("Jumlah testing :", len(X_test_text))

print("\nTRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

Jumlah training : 160
Jumlah testing : 40

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

TF-IDF digunakan untuk mengubah data teks menjadi bentuk numerik.

Proses `fit_transform()` dilakukan pada data training untuk membentuk vocabulary dan menghitung nilai TF-IDF.

Data testing menggunakan `transform()` berdasarkan vocabulary dari data training.

Parameter yang digunakan:

- `min_df=2`
- `max_df=0.95`

In [13]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

nama_fitur = tfidf.get_feature_names_out()

print("Jumlah fitur TF-IDF :", len(nama_fitur))
print("Shape training :", X_train_tfidf.shape)
print("Shape testing :", X_test_tfidf.shape)

Jumlah fitur TF-IDF : 3247
Shape training : (160, 3247)
Shape testing : (40, 3247)


## 14. Melihat Matriks TF-IDF

Hasil TF-IDF berupa matriks yang berisi bobot setiap kata pada setiap berita.

Setiap baris mewakili satu berita, sedangkan setiap kolom mewakili satu fitur atau kata.

In [14]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

display(tfidf_train_df.head())

,abad,abdullah,abdurrahman,absen,acara,acd,acosta,acuan,ada,adalah,...,yzr,zaki,zarco,zen,zero,zona,zone,zoom,zulhas,zulkifli
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.049264,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.080731,0.025628,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.056812,0.0,0.0,0.0,0.0,0.0,0.0,0.105606,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.053215,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.041250,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 15. Menyimpan Data TF-IDF

Hasil TF-IDF untuk data training dan testing disimpan dalam format Excel.

Kolom `label` ditambahkan sebagai kolom terakhir.

In [15]:
tfidf_train_df["label"] = y_train.reset_index(drop=True)

tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(drop=True)

tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

print("Data TF-IDF berhasil disimpan.")

Data TF-IDF berhasil disimpan.


## 16. Perhitungan ICSDF

ICSDF dihitung berdasarkan penyebaran suatu kata pada masing-masing kelas.

Pada dataset terdapat dua kelas, yaitu:

- Finance
- Sport

Untuk setiap kelas dihitung jumlah dokumen yang mengandung suatu kata. Nilai tersebut digunakan untuk mendapatkan kepadatan kata pada masing-masing kelas dan kemudian menghitung nilai ICSDF.

In [16]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

for kelas in sorted(y_train.unique()):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )

    jumlah_dokumen_class = (
        X_class.shape[0]
    )

    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )

    class_space_density += (
        class_density
    )

epsilon = 1e-12

icsdf = np.log(
    (jumlah_kelas + epsilon)
    /
    (class_space_density + epsilon)
)

df_icsdf = pd.DataFrame({
    "kata": features,
    "ICSDF": icsdf
})

display(
    df_icsdf.sort_values(
        "ICSDF",
        ascending=False
    ).head(20)
)

,kata,ICSDF
12,ade,4.382027
3246,zulkifli,4.382027
3245,zulhas,4.382027
3244,zoom,4.382027
2,abdurrahman,4.382027
18,adrenalin,4.382027
24,agostini,4.382027
3242,zona,4.382027
3241,zero,4.382027
1540,mantan,4.382027


## 17. TF-IDF × ICSDF

Nilai ICSDF digunakan untuk memberikan bobot tambahan pada matriks TF-IDF.

Proses dilakukan dengan mengalikan matriks TF-IDF dengan nilai ICSDF untuk data training dan testing.

In [17]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

print(
    "Shape training :",
    X_train_icsdf.shape
)

print(
    "Shape testing :",
    X_test_icsdf.shape
)

Shape training : (160, 3247)
Shape testing : (40, 3247)


## 18. Menentukan Kata Paling Penting

Setiap fitur diberikan skor berdasarkan nilai rata-ratanya pada data training.

Fitur kemudian diurutkan berdasarkan skor tersebut.

Fitur dengan skor tertinggi dianggap sebagai kata yang paling penting.

In [18]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(axis=0)
).ravel()

TOP_K = 100

top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

kata_penting = features[
    top_index
]

print(
    "Jumlah fitur terpilih :",
    len(kata_penting)
)

print("\nKata paling penting:")
print(kata_penting)

Jumlah fitur terpilih : 100

Kata paling penting:
['marquez' 'martin' 'emas' 'medali' 'gp' 'balapan' 'veda' 'airlangga' 'rp'
 'atlet' 'motogp' 'saya' 'pertandingan' 'saham' 'moto' 'asian' 'bpk'
 'maung' 'ketenagakerjaan' 'marc' 'dki' 'pasar' 'harga' 'triliun' 'gol'
 'industri' 'dpr' 'keuangan' 'kedelai' 'poin' 'detik' 'desak' 'negara'
 'ruu' 'ojk' 'undang' 'putra' 'mandalika' 'kawasan' 'kejuaraan' 'listrik'
 'rc' 'final' 'energi' 'posisi' 'anak' 'aset' 'perlindungan' 'novela'
 'hutan' 'kita' 'asuransi' 'bold' 'pekerja' 'jepang' 'perusahaan'
 'sudaryono' 'usaha' 'minyak' 'hilirisasi' 'jorge' 'menit' 'spa' 'riders'
 'bumn' 'prabowo' 'pilates' 'aditya' 'lap' 'produk' 'perak' 'putri'
 'beatrice' 'sepak' 'takraw' 'ri' 'rallycross' 'pebalap' 'sektor' 'laga'
 'panjat' 'indonesia' 'hektare' 'gaji' 'kg' 'aku' 'almansa' 'uu' 'set'
 'xi' 'tebing' 'ega' 'pemerintah' 'per' 'beras' 'nomor' 'dapur' 'umkm'
 'ya' 'investasi']


## 19. Reduksi Menjadi 100 Fitur ICSDF

Dari seluruh fitur yang tersedia, dipilih 100 fitur dengan skor tertinggi.

Matriks training dan testing kemudian direduksi menggunakan indeks fitur tersebut.

In [19]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

print(
    "Sebelum seleksi :",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF :",
    X_train_selected.shape
)

Sebelum seleksi : (160, 3247)
Sesudah ICSDF : (160, 100)


## 20. Tabel Hasil ICSDF

Hasil reduksi 100 fitur diubah menjadi DataFrame.

Nama kolom menggunakan 100 kata yang telah dipilih sebagai fitur penting.

In [20]:
icsdf_train_df = pd.DataFrame(
    X_train_selected.toarray(),
    columns=kata_penting
)

icsdf_train_df["label"] = (
    y_train.reset_index(drop=True)
)

display(
    icsdf_train_df.head()
)

,marquez,martin,emas,medali,gp,balapan,veda,airlangga,rp,atlet,...,ega,pemerintah,per,beras,nomor,dapur,umkm,ya,investasi,label
0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.084108,0.0,0.0,0.042054,0.0,0.0,0.000000,0.0,0
1,0.0,0.0,0.145621,0.206831,0.0,0.0,0.0,0.0,0.0,0.799055,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,1
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.042195,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0
3,0.0,0.0,0.211406,0.281502,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.044157,0.0,0.0,0.300488,0.0,1
4,0.0,0.0,0.000000,0.109955,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.043120,0.0,0.0,0.058685,0.0,1


In [21]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

print("Data ICSDF berhasil disimpan.")

Data ICSDF berhasil disimpan.


## 21. PCA

Setelah fitur direduksi menjadi 100 fitur menggunakan ICSDF, tahap selanjutnya adalah melakukan reduksi dimensi menggunakan Principal Component Analysis (PCA).

Pada tugas ini digunakan 20 komponen utama.

Dengan PCA, 100 fitur hasil ICSDF diubah menjadi 20 fitur baru yang disebut PC1 sampai PC20.

In [22]:
pca = PCA(
    n_components=20,
    random_state=42
)

X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

print(
    "Shape training PCA :",
    X_train_pca.shape
)

print(
    "Shape testing PCA :",
    X_test_pca.shape
)

Shape training PCA : (160, 20)
Shape testing PCA : (40, 20)


## 22. Membuat Tabel Data Reduksi Training

Hasil PCA pada data training dibuat menjadi DataFrame.

Data terdiri dari 20 komponen utama, yaitu PC1 sampai PC20.

Label ditambahkan sebagai kolom terakhir.

In [23]:
nama_pc = [
    f"PC{i}"
    for i in range(1, 21)
]

df_train_reduksi = pd.DataFrame(
    X_train_pca,
    columns=nama_pc
)

df_train_reduksi["label"] = (
    y_train.reset_index(drop=True)
)

display(
    df_train_reduksi.head()
)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.103473,0.016023,0.029519,-0.063367,-0.032107,0.048244,0.004501,0.031847,-0.026729,-0.000198,...,-0.132335,0.001120,-0.045458,-0.026296,-0.080432,-0.007706,-0.003137,0.089470,-0.023027,0
1,-0.141957,-0.214679,-0.114750,0.036311,-0.105307,-0.215132,0.041356,-0.021330,-0.004826,-0.012120,...,-0.061030,-0.305099,0.163021,-0.012305,0.011452,-0.011121,-0.029254,-0.008038,-0.011100,1
2,-0.065737,-0.007818,0.022237,-0.038576,-0.026585,0.022905,-0.003992,0.006766,-0.019636,0.009551,...,-0.068727,0.010519,-0.016516,0.000740,-0.009666,0.002551,-0.000990,0.026400,0.020223,0
3,-0.132712,-0.175759,-0.064482,-0.004610,-0.143851,-0.223778,0.037747,-0.021843,0.029612,-0.032285,...,0.085034,-0.065020,-0.089036,0.015802,0.006633,-0.018926,-0.127462,0.009556,0.051142,1
4,-0.126077,-0.253802,-0.085361,0.025473,-0.262903,-0.516720,-0.012022,-0.063083,-0.019598,-0.084147,...,0.327947,0.932703,-0.018361,-0.369887,0.026631,0.218790,1.472660,0.001989,0.007367,1


## 23. Membuat Data Testing

Data testing yang sebelumnya telah melalui proses TF-IDF dan ICSDF direduksi menggunakan PCA.

Data testing menggunakan transformasi PCA yang telah dibentuk berdasarkan data training.

Hasilnya terdiri dari 20 komponen utama dan satu kolom label.

In [24]:
df_test_reduksi = pd.DataFrame(
    X_test_pca,
    columns=nama_pc
)

df_test_reduksi["label"] = (
    y_test.reset_index(drop=True)
)

display(
    df_test_reduksi.head()
)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.169434,-0.024575,0.075555,-0.186312,-0.235520,0.353887,0.063091,0.060210,0.421983,-0.032524,...,0.032122,-0.008812,-0.075121,-0.047957,-0.068639,-0.032190,-0.027022,0.005854,-0.017872,0
1,-0.114563,-0.170591,-0.084798,0.032593,-0.100914,-0.202871,0.016181,-0.020897,-0.008401,-0.026728,...,0.059445,0.026756,-0.033831,0.008532,-0.001686,0.003863,0.016758,0.001457,-0.001811,1
2,-0.054209,-0.014835,0.034868,-0.008806,-0.022722,0.010414,-0.001631,0.008544,-0.009183,0.012789,...,-0.066308,0.004424,-0.000682,0.010735,-0.000797,0.005589,0.007221,0.011531,0.002414,1
3,-0.079778,-0.071291,-0.016151,0.004746,-0.058783,-0.070242,0.001563,-0.002179,-0.016713,-0.003245,...,-0.046489,-0.019185,0.018213,0.003720,0.003180,0.011677,0.067959,0.014946,-0.003106,1
4,-0.144207,0.008614,0.091111,-0.134713,-0.134532,0.269041,0.038544,0.001449,-0.028930,-0.014457,...,-0.352756,0.101344,-0.322381,-0.182229,-0.469877,-0.096961,-0.026766,-0.771438,-0.640385,0


## 24. Menyimpan Data Reduksi

Data hasil reduksi PCA untuk training dan testing disimpan dalam format Excel.

Setiap file berisi 20 komponen PCA dan satu kolom label.

In [25]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

print("Data reduksi berhasil disimpan.")

Data reduksi berhasil disimpan.


## 25. Cek Total Data Training dan Testing

Pemeriksaan dilakukan untuk memastikan jumlah data training dan testing sesuai dengan pembagian sebelumnya.

Data yang diharapkan:

- Training: 160 data
- Testing: 40 data
- Jumlah kolom: 21

Jumlah kolom terdiri dari:

- 20 fitur PCA
- 1 kolom label

In [26]:
print(
    "Shape training :",
    df_train_reduksi.shape
)

print(
    "Shape testing :",
    df_test_reduksi.shape
)

print(
    "\nJumlah training :",
    len(df_train_reduksi)
)

print(
    "Jumlah testing :",
    len(df_test_reduksi)
)

Shape training : (160, 21)
Shape testing : (40, 21)

Jumlah training : 160
Jumlah testing : 40


## 26. Memastikan Kolom Terakhir adalah Label

Pemeriksaan terakhir dilakukan untuk memastikan kolom `label` berada pada posisi paling akhir.

Dengan demikian data hasil reduksi memiliki 20 fitur PCA dan satu label sebagai kolom terakhir.

In [27]:
print("Kolom terakhir training :")
print(df_train_reduksi.columns[-1])

print("\nKolom terakhir testing :")
print(df_test_reduksi.columns[-1])

Kolom terakhir training :
label

Kolom terakhir testing :
label


In [28]:
print("=== PEMERIKSAAN AKHIR TUGAS 2 ===")

print("\nDataset awal :", df.shape)
print("Training PCA :", df_train_reduksi.shape)
print("Testing PCA  :", df_test_reduksi.shape)

print("\nJumlah fitur TF-IDF :", len(nama_fitur))
print("Jumlah fitur ICSDF :", len(kata_penting))
print("Jumlah komponen PCA :", len(nama_pc))

print("\nKolom terakhir training :", df_train_reduksi.columns[-1])
print("Kolom terakhir testing  :", df_test_reduksi.columns[-1])

print("\nFile yang telah dibuat:")
print("- data_tfidf_training.xlsx")
print("- data_tfidf_testing.xlsx")
print("- data_icsdf_training.xlsx")
print("- data_reduksi_training.xlsx")
print("- data_reduksi_testing.xlsx")

=== PEMERIKSAAN AKHIR TUGAS 2 ===

Dataset awal : (200, 7)
Training PCA : (160, 21)
Testing PCA  : (40, 21)

Jumlah fitur TF-IDF : 3247
Jumlah fitur ICSDF : 100
Jumlah komponen PCA : 20

Kolom terakhir training : label
Kolom terakhir testing  : label

File yang telah dibuat:
- data_tfidf_training.xlsx
- data_tfidf_testing.xlsx
- data_icsdf_training.xlsx
- data_reduksi_training.xlsx
- data_reduksi_testing.xlsx
